**Подготовка файла 2513.csv на основе данных сводных файлов 2016-2018 годов**

- Автор: Клинкова Екатерина
- Telegram: @ekaklinkova

- `..\Data\{год}\` - папка для хранения созданнного 2100.csv для определенного года
- `..\Data\{год}\Original` - папка для хранения оригинального файла xlsx
- `..\Data\Common\` - папка для хранения итогового результата 2100.csv сводный

Пример: 
- *E:/IrkutskProject/Data/2016/Original/ф33  правл за 2016г Иркутская обл.xlsx*
- *E:/IrkutskProject/Data/2016/2100.csv*

In [4]:
# Загрузка библиотек
import pandas as pd
import numpy as np
import os
import docx
import re

In [5]:
path = 'E:/IrkutskProject/Data/'
# Список годов для обработки
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

# Название оригинального файла
original_names = {
    2016: 'ИО 30 2016.docx',
    2017: 'ИО 30 2017.docx',
    2018: 'ИО 30 2018.docx',
    2019: 'ИО 30 2019.docx',
    2020: 'ИО 30 2020.docx',
    2021: '30 ИО 2021.docx',
    2022: '30 ИО 2022.docx',
    2023: '30 ИО 2023.docx',
    2024: '30 ИО 2024.docx'
}

original_url = {}
for year in years:
    original_path = f'{path}{year}/Original/'
    original_url[year] = original_path + original_names[year]
    
    if not os.path.exists(original_url[year]):
        print(f"Файл {original_url[year]} не найден по указанному пути")
    else:
        print(f"Файл {original_url[year]} найден по указанному пути")

Файл E:/IrkutskProject/Data/2016/Original/ИО 30 2016.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2017/Original/ИО 30 2017.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2018/Original/ИО 30 2018.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2019/Original/ИО 30 2019.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2020/Original/ИО 30 2020.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2021/Original/30 ИО 2021.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2022/Original/30 ИО 2022.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2023/Original/30 ИО 2023.docx найден по указанному пути
Файл E:/IrkutskProject/Data/2024/Original/30 ИО 2024.docx найден по указанному пути


In [6]:
def extract_tuberculosis_table(file_path, target_block):
    doc = docx.Document(file_path)
    target_found = False
    table_index = -1

    # Перебираем все элементы в теле документа
    for i, element in enumerate(doc.element.body):
        # Шаг 1: Ищем текст заголовка
        if not target_found and element.tag.endswith('p'):
            text = "".join(element.itertext()).strip()
            # display(text)
            if target_block.lower() in text.lower():
                target_found = True
                print(f"Текст '{target_block}' найден. Ищем следующую таблицу...")

        # Шаг 2: После нахождения текста ищем ПЕРВУЮ встречную таблицу
        if target_found and element.tag.endswith('tbl'):
            # Считаем количество таблиц до текущего элемента
            table_count = len([e for e in doc.element.body[:i] if e.tag.endswith('tbl')])
            table_index = table_count
            print(f"Таблица найдена (индекс в документе: {table_index})")
            break  # Выходим после нахождения первой таблицы

    if not target_found:
        raise ValueError(f"Текст '{target_block}' не найден в документе")
    elif table_index == -1:
        raise ValueError(f"Текст найден, но после него нет ни одной таблицы")

    # Возвращаем всю таблицу (включая все страницы)
    return doc.tables[table_index]

def table_to_dataframe(table):
    data = []
    for row in table.rows:
        current_row = []
        
        for cell in row.cells:
            text = cell.text.strip()
            current_row.append(text)
        data.append(current_row)

    # Создаём DataFrame
    if data:  # Проверяем, что таблица не пустая
        df = pd.DataFrame(data[1:], columns=data[0])
    else:
        df = pd.DataFrame()  # Пустой DataFrame, если таблица пустая
    return df

In [7]:
try:
    df_2513 = {}
    target_block = "2513"
    for year in years:
        display(year)
        table = extract_tuberculosis_table(original_url[year], target_block)
        df_2513[year] = table_to_dataframe(table)
        df_2513[year].to_csv(f'E:/IrkutskProject/Data/{year}/2513.csv', index=False, encoding='utf-8-sig')
except Exception as e:
    print(f"Произошла ошибка: {str(e)}")

2016

Текст '2513' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 75)


2017

Текст '2513' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 75)


2018

...

Текст '2513' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 74)


2023

Текст '2513' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 74)


2024

Текст '2513' найден. Ищем следующую таблицу...
Таблица найдена (индекс в документе: 78)


In [8]:
for year in years:
    display(df_2513[year])

,Профилактические осмотры на туберкулез,№\nстроки,Всего,из них \nсельских жителей,Выявлен туберкулез,Выявлен туберкулез
0,Профилактические осмотры на туберкулез,№\nстроки,Всего,из них \nсельских жителей,Всего,из них:\nу сельских жителей
1,1,2,3,4,5,6
2,"Осмотрено пациентов, всего",1,1892276,389647,1654,469
3,из них детей: 1-7 лет включительно,1.1,248618,63399,57,14
4,8-14 лет включительно,1.2,204640,46759,27,10
5,15-17 лет включительно,1.3,77945,17127,44,7
6,Из числа осмотренных (стр.1) обследовано:\nфлю...,2,1411232,275966,1523,415
7,бактериоскопически,3,11666,2353,40,9
8,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,,,,,
9,иммунодиагностика с применением аллергена бакт...,4,455644,111134,83,24


,Профилактические осмотры на туберкулез,№\nстроки,Всего,из них \nсельских жителей,Выявлен туберкулез,Выявлен туберкулез
0,Профилактические осмотры на туберкулез,№\nстроки,Всего,из них \nсельских жителей,Всего,из них:\nу сельских жителей
1,1,2,3,4,5,6
2,"Осмотрено пациентов, всего",1,1976734,393076,1463,405
3,из них детей: 1-7 лет включительно,1.1,254999,53259,42,15
4,8-14 лет включительно,1.2,212703,47101,16,6
5,15-17 лет включительно,1.3,74295,16894,15,6
6,Из числа осмотренных (стр.1) обследовано:\nфлю...,2,1504164,290241,1386,380
7,бактериоскопически,3,6379,1454,18,4
8,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,,,,,
9,иммунодиагностика с применением аллергена бакт...,4,468160,100780,59,21


,Профилактические осмотры на туберкулез,№\nстроки,"Всего, чел",из них \nсельских жителей,"Выявлен туберкулез, \nвключая рецидив туберкулеза","Выявлен туберкулез, \nвключая рецидив туберкулеза"
0,Профилактические осмотры на туберкулез,№\nстроки,"Всего, чел",из них \nсельских жителей,Всего,из них\nу сельских жителей
1,1,2,3,4,5,6
2,"Осмотрено пациентов, всего",1,1962309,422308,1257,357
3,из них детей: 0-7 лет включительно,1.1,246373,55765,34,7
4,8-14 лет включительно,1.2,218768,47755,12,4
5,15-17 лет включительно,1.3,82860,18019,16,5
6,Из числа осмотренных (стр.1) обследовано:\n ...,2,1488995,316851,1204,339
7,бактериоскопически,3,6794,1594,7,7
8,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,4,270148,63276,32,9
9,иммунодиагностика с применением аллергена тубе...,5,195927,40398,14,2


,Профилактические осмотры на туберкулез,№\nстроки,Всего,из них \nсельских жителей,Выявлен туберкулез,Выявлен туберкулез
0,Профилактические осмотры на туберкулез,№\nстроки,Всего,из них \nсельских жителей,Всего,из них:\nу сельских жителей
1,1,2,3,4,5,6
2,"Осмотрено пациентов, всего",1,1998055,415453,1171,339
3,из них детей: 1-7 лет включительно,1.1,248633,55668,18,2
4,8-14 лет включительно,1.2,226578,49134,11,2
5,15-17 лет включительно,1.3,86209,16827,21,7
6,Из числа осмотренных (стр.1) обследовано:\nфлю...,2,1518209,309277,1129,327
7,бактериоскопически,3,4525,1374,12,7
8,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,,,,,
9,иммунодиагностика с применением аллергена бакт...,4,248633,55587,19,2


,Профилактические осмотры на туберкулез,№\nстроки,"Всего, чел",из них \nсельских жителей,"Выявлен туберкулез, \nвключая рецидив туберкулеза","Выявлен туберкулез, \nвключая рецидив туберкулеза"
0,Профилактические осмотры на туберкулез,№\nстроки,"Всего, чел",из них \nсельских жителей,Всего,из них\nу сельских жителей
1,1,2,3,4,5,6
2,"Осмотрено пациентов, всего",1,1707268,350018,895,257
3,из них детей: 0-7 лет включительно,1.1,218293,47990,9,1
4,8-14 лет включительно,1.2,208833,47180,13,6
5,15-17 лет включительно,1.3,77038,16131,13,7
6,Из числа осмотренных (стр.1) обследовано:\n ...,2,1276013,253509,829,241
7,бактериоскопически,3,3773,1304,44,9
8,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,4,218293,47990,9,1
9,иммунодиагностика с применением аллергена тубе...,5,209180,47212,13,6


,Профилактические осмотры на туберкулез,№\nстроки,Всего,из них \nсельских жителей,Выявлен туберкулез,Выявлен туберкулез
0,Профилактические осмотры на туберкулез,№\nстроки,Всего,из них \nсельских жителей,Всего,из них:\nу сельских жителей
1,1,2,3,4,5,6
2,"Осмотрено пациентов, всего",1,1849615,375787,824,214
3,из них детей: 1-7 лет включительно,1.1,220294,47465,12,1
4,8-14 лет включительно,1.2,227968,50682,8,3
5,15-17 лет включительно,1.3,83434,18792,7,
6,Из числа осмотренных (стр.1) обследовано:\nфлю...,2,1391224,276489,772,203
7,бактериоскопически,3,3593,967,29,7
8,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,,,,,
9,иммунодиагностика с применением аллергена бакт...,4,220294,47465,12,1


,Профилактические осмотры на туберкулез,№\nстроки,"Всего, чел",из них \nсельских жителей,Выявлен туберкулез,Выявлен туберкулез
0,Профилактические осмотры на туберкулез,№\nстроки,"Всего, чел",из них \nсельских жителей,Всего,из них\nу сельских жителей
1,1,2,3,4,5,6
2,"Осмотрено пациентов, всего",1,1895343,387147,825,227
3,из них детей: 1-7 лет включительно,1.1,215589,45470,10,5
4,8-14 лет включительно,1.2,240553,50896,8,4
5,15-17 лет включительно,1.3,85743,19398,14,3
6,Из числа осмотренных (стр.1) обследовано:\n ...,2,1431385,301410,766,198
7,бактериоскопически,3,4005,1296,41,20
8,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,,,,,
9,иммунодиагностика с применением аллергена бакт...,4,215589,45470,10,4


,Профилактические осмотры на туберкулез,№\nстроки,"Всего, чел",из них \nсельских жителей,Выявлен туберкулез,Выявлен туберкулез
0,Профилактические осмотры на туберкулез,№\nстроки,"Всего, чел",из них \nсельских жителей,Всего,из них\nу сельских жителей
1,1,2,3,4,5,6
2,"Осмотрено пациентов, всего",1,1910125,428727,782,220
3,из них детей: 1-7 лет включительно,1.1,219146,51667,9,3
4,8-14 лет включительно,1.2,237892,57116,9,4
5,15-17 лет включительно,1.3,87439,22347,8,1
6,Из числа осмотренных (стр.1) обследовано:\n ...,2,1444938,318128,745,208
7,бактериоскопически,3,3441,682,19,5
8,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,,,,,
9,иммунодиагностика с применением аллергена бакт...,4,219146,51667,10,4


,Профилактические осмотры на туберкулез,№\nстроки,"Всего, чел",из них \nсельских жителей,Выявлен туберкулез,Выявлен туберкулез
0,Профилактические осмотры на туберкулез,№\nстроки,"Всего, чел",из них \nсельских жителей,Всего,из них\nу сельских жителей
1,1,2,3,4,5,6
2,"Осмотрено пациентов, всего",1,1979761,465566,743,230
3,из них детей: 0-7 лет включительно,1.1,210816,53781,6,5
4,8-14 лет включительно,1.2,249925,64357,5,4
5,15-17 лет включительно,1.3,94260,24350,12,4
6,Из числа осмотренных (стр.1) обследовано:\n ...,2,1517149,348500,699,205
7,бактериоскопически,3,2731,478,33,16
8,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,4,210816,53781,5,5
9,иммунодиагностика с применением аллергена тубе...,5,250750,64512,6,4


In [9]:
for year in years:
    if (year != 2018) and (year != 2020) and (year != 2024):
        df_2513[year].iloc[9,0] = 'Из числа осмотренных детей (стр. 1.1+1.2+1.3) проведены ' + df_2513[year].iloc[9,0].strip()
        df_2513[year].drop(df_2513[year].index[8:9], inplace=True)
    df_2513[year].iloc[6,0] = df_2513[year].iloc[6,0].replace(':\n', ' ')

    df_2513[year].iloc[6,0] = df_2513[year].iloc[6,0].replace('              ', ' ')
    df_2513[year].iloc[8,0] = df_2513[year].iloc[8,0].replace(':', '')
    df_2513[year].iloc[8,0] = df_2513[year].iloc[8,0].replace('\n', '')
    df_2513[year].iloc[8,0] = df_2513[year].iloc[8,0].replace('  ',' ')
    df_2513[year].iloc[7,0] = 'Из числа осмотренных (стр.1) обследовано ' + df_2513[year].iloc[7,0].strip()
    df_2513[year].iloc[9,0] = 'Из числа осмотренных детей (стр. 1.1+1.2+1.3) проведены ' + df_2513[year].iloc[9,0].strip()

    df_2513[year].columns = [
        'Профилактические осмотры на туберкулез',
        '№ строки',
        'Всего',
        'из них сельских жителей',
        'Выявлен туберкулез',
        'из них: у сельских жителей'
    ]
    
    df_2513[year] = df_2513[year].drop(df_2513[year].columns[5:], axis=1)
    df_2513[year] = df_2513[year].drop(df_2513[year].columns[3:4], axis=1)
    df_2513[year].drop(df_2513[year].index[10:11], inplace=True)
    df_2513[year].drop(df_2513[year].index[3:6], inplace=True)
    df_2513[year].drop(df_2513[year].index[0:1], inplace=True)
    #df_2513[year] = df_2513[year].reset_index(drop=True)

    

In [10]:
for year in years:
    display(df_2513[year])

,Профилактические осмотры на туберкулез,№ строки,Всего,Выявлен туберкулез
1,1,2,3,5
2,"Осмотрено пациентов, всего",1,1892276,1654
6,Из числа осмотренных (стр.1) обследовано флюор...,2,1411232,1523
7,Из числа осмотренных (стр.1) обследовано бакте...,3,11666,40
9,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,4,455644,83
10,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,5,,


,Профилактические осмотры на туберкулез,№ строки,Всего,Выявлен туберкулез
1,1,2,3,5
2,"Осмотрено пациентов, всего",1,1976734,1463
6,Из числа осмотренных (стр.1) обследовано флюор...,2,1504164,1386
7,Из числа осмотренных (стр.1) обследовано бакте...,3,6379,18
9,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,4,468160,59
10,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,5,861,


,Профилактические осмотры на туберкулез,№ строки,Всего,Выявлен туберкулез
1,1,2,3,5
2,"Осмотрено пациентов, всего",1,1962309,1257
6,Из числа осмотренных (стр.1) обследовано флюор...,2,1488995,1204
7,Из числа осмотренных (стр.1) обследовано бакте...,3,6794,7
8,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,4,270148,32
9,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,5,195927,14


,Профилактические осмотры на туберкулез,№ строки,Всего,Выявлен туберкулез
1,1,2,3,5
2,"Осмотрено пациентов, всего",1,1998055,1171
6,Из числа осмотренных (стр.1) обследовано флюор...,2,1518209,1129
7,Из числа осмотренных (стр.1) обследовано бакте...,3,4525,12
9,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,4,248633,19
10,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,5,226578,10


,Профилактические осмотры на туберкулез,№ строки,Всего,Выявлен туберкулез
1,1,2,3,5
2,"Осмотрено пациентов, всего",1,1707268,895
6,Из числа осмотренных (стр.1) обследовано флюор...,2,1276013,829
7,Из числа осмотренных (стр.1) обследовано бакте...,3,3773,44
8,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,4,218293,9
9,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,5,209180,13


,Профилактические осмотры на туберкулез,№ строки,Всего,Выявлен туберкулез
1,1,2,3,5
2,"Осмотрено пациентов, всего",1,1849615,824
6,Из числа осмотренных (стр.1) обследовано флюор...,2,1391224,772
7,Из числа осмотренных (стр.1) обследовано бакте...,3,3593,29
9,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,4,220294,12
10,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,5,228354,8


,Профилактические осмотры на туберкулез,№ строки,Всего,Выявлен туберкулез
1,1,2,3,5
2,"Осмотрено пациентов, всего",1,1895343,825
6,Из числа осмотренных (стр.1) обследовано флюор...,2,1431385,766
7,Из числа осмотренных (стр.1) обследовано бакте...,3,4005,41
9,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,4,215589,10
10,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,5,241133,7


,Профилактические осмотры на туберкулез,№ строки,Всего,Выявлен туберкулез
1,1,2,3,5
2,"Осмотрено пациентов, всего",1,1910125,782
6,Из числа осмотренных (стр.1) обследовано флюор...,2,1444938,745
7,Из числа осмотренных (стр.1) обследовано бакте...,3,3441,19
9,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,4,219146,10
10,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,5,239459,8


,Профилактические осмотры на туберкулез,№ строки,Всего,Выявлен туберкулез
1,1,2,3,5
2,"Осмотрено пациентов, всего",1,1979761,743
6,Из числа осмотренных (стр.1) обследовано флюор...,2,1517149,699
7,Из числа осмотренных (стр.1) обследовано бакте...,3,2731,33
8,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,4,210816,5
9,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,5,250750,6


In [11]:
columns = [
    'Показатель',
    'Уточнение',
    'Год',
    'Значение',
    'Строка',
    'Графа',
    'Таблица'
]

data = []
data_2016_2018 = []

df_2513_long = pd.DataFrame(columns=columns)
df_2513_long_2016_2018 = pd.DataFrame(columns=columns)

for year in years:
    for col_number in range(2, df_2513[year].shape[1]):
        for row_number in range(1, df_2513[year].shape[0]):
            if (
                ((df_2513[year].iloc[row_number,1] == '1') and (df_2513[year].iloc[0, col_number] == '5')) or
                ((df_2513[year].iloc[row_number,1] == '4') and (df_2513[year].iloc[0, col_number] == '5')) or
                ((df_2513[year].iloc[row_number,1] == '5') and (df_2513[year].iloc[0, col_number] == '5'))
            ):
                continue
            else:
                # print(df_2513[year].iloc[row_number,1], df_2513[year].iloc[0, col_number])
                new_row = []
                new_row.append(df_2513[year].iloc[row_number, 0]) # Показатель
                new_row.append(df_2513[year].columns[col_number]) # Уточнение
                new_row.append(year)
                new_row.append(df_2513[year].iloc[row_number, col_number]) # Значение
                new_row.append(df_2513[year].iloc[row_number,1]) # Строка
                new_row.append(df_2513[year].iloc[0, col_number]) # Графа
                new_row.append(2513)
                data.append(new_row)
                if year <= 2018:
                   data_2016_2018.append(new_row)

df_2513_long = pd.DataFrame(data, columns=columns)
df_2513_long_2016_2018 = pd.DataFrame(data_2016_2018, columns=columns)


In [12]:
display(df_2513_long_2016_2018)

,Показатель,Уточнение,Год,Значение,Строка,Графа,Таблица
0,"Осмотрено пациентов, всего",Всего,2016,1892276,1,3,2513
1,Из числа осмотренных (стр.1) обследовано флюор...,Всего,2016,1411232,2,3,2513
2,Из числа осмотренных (стр.1) обследовано бакте...,Всего,2016,11666,3,3,2513
3,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,Всего,2016,455644,4,3,2513
4,Из числа осмотренных детей (стр. 1.1+1.2+1.3) ...,Всего,2016,,5,3,2513
5,Из числа осмотренных (стр.1) обследовано флюор...,Выявлен туберкулез,2016,1523,2,5,2513
6,Из числа осмотренных (стр.1) обследовано бакте...,Выявлен туберкулез,2016,40,3,5,2513
7,"Осмотрено пациентов, всего",Всего,2017,1976734,1,3,2513
8,Из числа осмотренных (стр.1) обследовано флюор...,Всего,2017,1504164,2,3,2513
9,Из числа осмотренных (стр.1) обследовано бакте...,Всего,2017,6379,3,3,2513


In [13]:
df_2513_long_2016_2018.to_csv(f'{path}Common_2016-2018/2513_2016-2018.csv', index=False)
df_2513_long.to_csv(f'{path}Common/2513_2016-2024.csv', index=False)